# Build Clean Demographics And Message-Time Charlson CCI Data

Purpose: create clean demographic variables and Charlson Comorbidity Index (CCI) variables using the same problem-list timing logic used in the analytic preprocessing pipeline.

This notebook focuses on demographics and CCI only. It does not build visit/encounter variables, construct threads, or run regression models.

Design notes:

- `Demog.txt` provides patient gender, race, and ethnicity.
- `BirthYear.xlsx` provides birth year for age-at-message calculations after merge.
- `ProblemList.txt` provides ICD-10 diagnosis history with noted/resolved dates.
- `Charlson_comorbidity_index.csv` provides the ICD-10 to Charlson category/weight map.
- `messages_without_threads.parquet` provides the message IDs, patient IDs, and message timestamps used to compute message-time CCI.
- CCI is computed at each message timestamp using active diagnoses: `NOTED_DATE <= CREATED_TIME` and `RESOLVED_DATE` missing or `RESOLVED_DATE >= CREATED_TIME`.
- No row-level patient or message examples are displayed.


## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import re

import numpy as np
import pandas as pd

pd.options.display.max_columns = 140


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


def required_restricted_data_dir() -> Path:
    value = os.environ.get("JMIR_RESTRICTED_DATA_DIR")
    if not value:
        raise EnvironmentError(
            "Set JMIR_RESTRICTED_DATA_DIR to the authorized local data root before running. "
            "See data/README.md for the expected directory layout."
        )
    return Path(value).expanduser().resolve()


REPO_ROOT = find_repo_root()
OUTPUT_ROOT = Path(os.environ.get("JMIR_REPRO_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
RUNS_DIR = OUTPUT_ROOT / "runs"
WORKING_OUTPUT_DIR = Path(
    os.environ.get("JMIR_WORKING_DATA_DIR", OUTPUT_ROOT / "working_data")
).expanduser().resolve()

RESTRICTED_DATA_DIR = required_restricted_data_dir()
SOURCE_RAW_DIR = Path(
    os.environ.get("JMIR_RAW_DATA_DIR", RESTRICTED_DATA_DIR / "raw")
).expanduser().resolve()
SOURCE_ADDED_DIR = Path(
    os.environ.get("JMIR_ADDITIONAL_DATA_DIR", SOURCE_RAW_DIR / "added October 25")
).expanduser().resolve()
SOURCE_CLEANED_DIR = Path(
    os.environ.get("JMIR_CLEANED_MESSAGE_DIR", RESTRICTED_DATA_DIR / "Cleaned")
).expanduser().resolve()
MESSAGE_SOURCE_DIR = WORKING_OUTPUT_DIR


def resolve_message_input(file_name: str) -> Path:
    path = MESSAGE_SOURCE_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(
            f"Missing message handoff: {path}\n"
            "Run 01_build_core_message_preprocessing_clean.ipynb first."
        )
    return path


DEMOG_PATH = SOURCE_RAW_DIR / "Demog.txt"
BIRTH_YEAR_PATH = SOURCE_ADDED_DIR / "BirthYear.xlsx"
PROBLEM_LIST_PATH = SOURCE_RAW_DIR / "ProblemList.txt"
CHARLSON_LOOKUP_PATH = SOURCE_CLEANED_DIR / "Charlson_comorbidity_index.csv"
MESSAGE_DATA_PATH = resolve_message_input("messages_without_threads.parquet")
CANONICAL_PATIENT_DEMOGRAPHICS_PATH = MESSAGE_SOURCE_DIR / "patient_demographics_clean.parquet"
CANONICAL_MESSAGE_DEMOGRAPHICS_PATH = MESSAGE_SOURCE_DIR / "message_demographics_clean.parquet"
CANONICAL_MESSAGE_CHARLSON_CCI_PATH = MESSAGE_SOURCE_DIR / "message_level_charlson_cci.parquet"

MESSAGE_ID_COL = "MESSAGE_ID"
MESSAGE_DMRN_COL = "DMRN"
MESSAGE_TIME_COL = "CREATED_TIME"
MESSAGE_REQUIRED_COLS = [MESSAGE_ID_COL, MESSAGE_DMRN_COL, MESSAGE_TIME_COL]

RUNS_DIR.mkdir(parents=True, exist_ok=True)
WORKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUNS_DIR / f"demographics_cci_{RUN_ID}"
if RUN_DIR.exists():
    raise FileExistsError(f"Run directory already exists: {RUN_DIR}")
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("REPO_ROOT:", REPO_ROOT)
print("RUN_DIR:", RUN_DIR)
print("SOURCE_RAW_DIR:", SOURCE_RAW_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("WORKING_OUTPUT_DIR:", WORKING_OUTPUT_DIR)
print("CHARLSON_LOOKUP_PATH:", CHARLSON_LOOKUP_PATH)
print("MESSAGE_DATA_PATH:", MESSAGE_DATA_PATH)
print("CANONICAL_MESSAGE_DEMOGRAPHICS_PATH:", CANONICAL_MESSAGE_DEMOGRAPHICS_PATH)
print("CANONICAL_MESSAGE_CHARLSON_CCI_PATH:", CANONICAL_MESSAGE_CHARLSON_CCI_PATH)

## 2. Helper Functions

In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.encode("ascii", "ignore")
        .str.decode("ascii")
        .str.strip()
        .str.replace(r"[^\w]+", "_", regex=True)
    )
    return out


def read_source_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    else:
        sep = "	" if path.suffix.lower() == ".txt" else ","
        df = pd.read_csv(
            path,
            encoding="cp1252",
            encoding_errors="replace",
            sep=sep,
            engine="python",
            on_bad_lines="skip",
        )
    return clean_columns(df)


def clean_id(s):
    return s.astype("string").str.strip()


def to_ts(s):
    out = pd.to_datetime(s, errors="coerce", utc=False)
    try:
        if getattr(out.dt, "tz", None) is not None:
            out = out.dt.tz_convert(None)
    except Exception:
        pass
    return out


def normalize_icd(code):
    if pd.isna(code):
        return pd.NA
    return re.sub(r"[^A-Z0-9]", "", str(code).upper().strip())


def join_unique(values):
    vals = pd.Series(values).dropna().astype(str).str.strip()
    vals = vals.loc[vals.ne("")]
    if vals.empty:
        return pd.NA
    return "; ".join(sorted(vals.drop_duplicates()))


## 3. Load Source Tables

In [ ]:
Demog = read_source_table(DEMOG_PATH)
BirthYear = read_source_table(BIRTH_YEAR_PATH)
ProblemList = read_source_table(PROBLEM_LIST_PATH)
CharlsonLookupRaw = read_source_table(CHARLSON_LOOKUP_PATH)

if not MESSAGE_DATA_PATH.exists():
    raise FileNotFoundError(MESSAGE_DATA_PATH)
MessagesForCCI = pd.read_parquet(MESSAGE_DATA_PATH, columns=MESSAGE_REQUIRED_COLS)
MessagesForCCI = clean_columns(MessagesForCCI)
missing_message_cols = sorted(set(MESSAGE_REQUIRED_COLS) - set(MessagesForCCI.columns))
if missing_message_cols:
    raise ValueError(f"MESSAGE_DATA_PATH is missing required columns: {missing_message_cols}")

for name, df in {
    "Demog": Demog,
    "BirthYear": BirthYear,
    "ProblemList": ProblemList,
    "CharlsonLookupRaw": CharlsonLookupRaw,
    "MessagesForCCI": MessagesForCCI,
}.items():
    print(f"{name}: {len(df):,} rows, {df.shape[1]} columns")
    print("  columns:", list(df.columns))


## 4. Build Patient And Message-Level Demographics


In [ ]:
demog = Demog.copy()
birth_year = BirthYear.copy()

demog["DMRN"] = clean_id(demog["DMRN"])
birth_year["DMRN"] = clean_id(birth_year["DMRN"])
birth_year["BIRTH_YEAR"] = pd.to_numeric(birth_year["BIRTH_YEAR"], errors="coerce").astype("Int64")

demog = demog.dropna(subset=["DMRN"]).drop_duplicates("DMRN").copy()
demog["Gender"] = demog["Gender"].astype("string").str.strip().str.title()
demog["Race"] = demog["Race"].astype("string").str.strip()
demog["Ethnicity"] = demog["Ethnicity"].astype("string").str.strip()

race_map = {
    "WHITE": "White",
    "AFRICAN AMERICAN/BLK": "Black/African American",
    "AFRICAN OTHER": "Black/African American",
    "SOMALI": "Black/African American",
    "ASIAN CAMBODIAN": "Asian",
    "ASIAN CHINESE": "Asian",
    "ASIAN FILIPINO": "Asian",
    "ASIAN INDIAN": "Asian",
    "ASIAN JAPANESE": "Asian",
    "ASIAN KOREAN": "Asian",
    "ASIAN LAOTIAN": "Asian",
    "ASIAN OTHER": "Asian",
    "ASIAN PAKISTANI": "Asian",
    "ASIAN VIETNAMESE": "Asian",
    "NEPALI": "Asian",
    "MIDDLE EASTERN": "Middle Eastern/North African",
    "MIDDLE EASTERN NORTHERN AFRICAN - EGYPTIAN": "Middle Eastern/North African",
    "MIDDLE EASTERN NORTHERN AFRICAN - IRANIAN": "Middle Eastern/North African",
    "MIDDLE EASTERN NORTHERN AFRICAN - IRAQI": "Middle Eastern/North African",
    "MIDDLE EASTERN NORTHERN AFRICAN - ISRAELI": "Middle Eastern/North African",
    "MIDDLE EASTERN NORTHERN AFRICAN - LEBANESE": "Middle Eastern/North African",
    "MIDDLE EASTERN NORTHERN AFRICAN - OTHER": "Middle Eastern/North African",
    "MIDDLE EASTERN NORTHERN AFRICAN - SYRIAN": "Middle Eastern/North African",
    "AMER INDIAN/ALASKAN": "Native American/Alaskan",
    "GUAMANIAN OR CHAMORRO": "Pacific Islander",
    "NATIVE HAWAIIAN": "Pacific Islander",
    "NTV HAWAIIAN/PCF ISL": "Pacific Islander",
    "OTHER PACIFIC ISLANDER": "Pacific Islander",
    "MORE THAN ONE RACE": "Other/Multiple/Unknown",
    "OTHER": "Other/Multiple/Unknown",
    "PT NOT AVAILABLE": "Other/Multiple/Unknown",
    "REFUSED TO ANSWER": "Other/Multiple/Unknown",
    "UNKNOWN (TO PATIENT)": "Other/Multiple/Unknown",
    "MISSING VALUE": "Other/Multiple/Unknown",
}

ethnicity_map = {
    "LATINO/HISPANIC OTHER": "Hispanic/Latino",
    "MEXICAN, MEXICAN-AMER, CHICAN": "Hispanic/Latino",
    "PUERTO RICAN": "Hispanic/Latino",
    "CUBAN": "Hispanic/Latino",
    "NOT HISPANIC OR LATINO": "Not Hispanic/Latino",
    "ASHKENAZI JEW": "Not Hispanic/Latino",
    "PATIENT NOT AVAILABLE": "Unknown/Other",
    "REFUSED TO ANSWER": "Unknown/Other",
    "UNKNOWN (TO PATIENT)": "Unknown/Other",
    "OTHER": "Unknown/Other",
    "MISSING VALUE": "Unknown/Other",
}

demog["Race_Grouped"] = demog["Race"].astype("string").str.upper().map(race_map).fillna("Other/Multiple/Unknown")
demog["Ethnicity_Grouped"] = demog["Ethnicity"].astype("string").str.upper().map(ethnicity_map).fillna("Unknown/Other")

patient_demographics = (
    demog[["DMRN", "Gender", "Race", "Race_Grouped", "Ethnicity", "Ethnicity_Grouped"]]
    .merge(birth_year[["DMRN", "BIRTH_YEAR"]].drop_duplicates("DMRN"), on="DMRN", how="left")
    .drop_duplicates("DMRN")
    .reset_index(drop=True)
)

# Message-level demographics preserve the original analytic age logic: age is calculated at the message year.
message_demographics = MessagesForCCI.copy()
message_demographics["DMRN"] = clean_id(message_demographics["DMRN"])
message_demographics["CREATED_TIME"] = to_ts(message_demographics["CREATED_TIME"])
message_demographics = message_demographics.merge(
    birth_year[["DMRN", "BIRTH_YEAR"]].drop_duplicates("DMRN"),
    on="DMRN",
    how="left",
)
message_demographics["BIRTH_YEAR"] = pd.to_numeric(message_demographics["BIRTH_YEAR"], errors="coerce").astype("Int64")
message_demographics["Message_Year"] = message_demographics["CREATED_TIME"].dt.year.astype("Int64")
message_demographics["Age_at_Message"] = (
    message_demographics["Message_Year"] - message_demographics["BIRTH_YEAR"]
).astype("Int64")
# Preserve missing age instead of classifying it as younger than 65.
message_demographics["AgeGroup65"] = (
    message_demographics["Age_at_Message"] >= 65
).astype("Int64")

print("patient_demographics rows:", len(patient_demographics))
print("unique DMRN:", patient_demographics["DMRN"].nunique())
print("rows with BIRTH_YEAR:", patient_demographics["BIRTH_YEAR"].notna().sum())
print("message_demographics rows:", len(message_demographics))
print("messages with BIRTH_YEAR:", message_demographics["BIRTH_YEAR"].notna().sum())
print("messages with Age_at_Message:", message_demographics["Age_at_Message"].notna().sum())
print()
print(patient_demographics["Gender"].value_counts(dropna=False).to_string())
print()
print(patient_demographics["Race_Grouped"].value_counts(dropna=False).to_string())
print()
print(patient_demographics["Ethnicity_Grouped"].value_counts(dropna=False).to_string())


## 5. Clean Problem List And Charlson Lookup

In [ ]:
problem_list_clean = ProblemList.copy()
problem_list_clean["DMRN"] = clean_id(problem_list_clean["DMRN"])
problem_list_clean["NOTED_DATE"] = to_ts(problem_list_clean["NOTED_DATE"])
problem_list_clean["RESOLVED_DATE"] = to_ts(problem_list_clean["RESOLVED_DATE"])
problem_list_clean["DX_CODE"] = problem_list_clean["DX_CODE"].astype("string").str.upper().str.strip()
problem_list_clean = (
    problem_list_clean
    .dropna(subset=["DMRN", "DX_CODE"])
    .drop_duplicates()
    .reset_index(drop=True)
)

# Restore the Charlson lookup column names used in the original preprocessing code.
Charlson_comorbidity = CharlsonLookupRaw.copy().rename(columns={
    "Comorbidity_Category": "Comorbidity Category",
    "ICD_10_Code": "ICD-10 Code",
})
Charlson_comorbidity["ICD-10 Code"] = Charlson_comorbidity["ICD-10 Code"].astype("string").str.upper().str.strip()
Charlson_comorbidity["Comorbidity Category"] = Charlson_comorbidity["Comorbidity Category"].astype("string").str.strip()
Charlson_comorbidity["Weight"] = pd.to_numeric(Charlson_comorbidity["Weight"], errors="coerce").fillna(0).astype(int)

print("problem_list_clean rows:", len(problem_list_clean))
print("problem_list_clean patients:", problem_list_clean["DMRN"].nunique())
print("Charlson_comorbidity lookup rows:", len(Charlson_comorbidity))
print("Charlson_comorbidity columns:", list(Charlson_comorbidity.columns))


## 6. Map Problem-List Diagnoses To Charlson Categories

In [ ]:
def expand_charlson_fast(problem: pd.DataFrame, cci_map: pd.DataFrame) -> pd.DataFrame:
    """
    Map problem-list ICD-10 diagnosis codes to Charlson categories.

    This preserves the original analytic logic: exact code matches, `.x` prefix
    matches, and en-dash ranges from the Charlson lookup table.
    """
    df = problem.copy()
    df["DX_CODE"] = df["DX_CODE"].astype(str).str.upper().str.strip()

    cci = cci_map.copy()
    cci["ICD-10 Code"] = cci["ICD-10 Code"].astype(str).str.upper().str.strip()

    expanded = []

    def normalize_icd_for_range(code):
        return re.sub(r"[^A-Z0-9]", "", str(code).upper().strip())

    df["DX_NUM"] = df["DX_CODE"].apply(normalize_icd_for_range)

    for _, row in cci.iterrows():
        pattern = row["ICD-10 Code"]
        category = row["Comorbidity Category"]
        weight = row["Weight"]

        range_sep = "\u2013" if "\u2013" in pattern else "-" if "-" in pattern else None

        if ".X" not in pattern and range_sep is None:
            matched = df[df["DX_CODE"] == pattern]
        elif pattern.endswith(".X"):
            base = pattern[:-2]
            matched = df[df["DX_CODE"].str.startswith(base)]
        elif range_sep is not None:
            start, end = pattern.split(range_sep, 1)
            start_norm = normalize_icd_for_range(start)
            end_norm = normalize_icd_for_range(end)
            matched = df[(df["DX_NUM"] >= start_norm) & (df["DX_NUM"] <= end_norm)]
        else:
            matched = pd.DataFrame(columns=df.columns)

        if not matched.empty:
            matched = matched.assign(
                Charlson_category=category,
                Weight=weight,
            )
            expanded.append(matched)

    if not expanded:
        return pd.DataFrame(columns=["DMRN", "NOTED_DATE", "RESOLVED_DATE", "DX_CODE", "Charlson_category", "Weight"])

    problems_expanded = pd.concat(expanded, ignore_index=True)
    problems_expanded = problems_expanded.drop_duplicates(
        subset=["DMRN", "NOTED_DATE", "RESOLVED_DATE", "DX_CODE", "Charlson_category"]
    )
    return problems_expanded[["DMRN", "NOTED_DATE", "RESOLVED_DATE", "DX_CODE", "Charlson_category", "Weight"]]


problem_charlson_mapped = expand_charlson_fast(problem_list_clean, Charlson_comorbidity)
problem_charlson_mapped["Weight"] = pd.to_numeric(problem_charlson_mapped["Weight"], errors="coerce").fillna(0).astype(int)

patient_charlson_ever = (
    problem_charlson_mapped
    .drop_duplicates(["DMRN", "Charlson_category"])
    .groupby("DMRN", as_index=False)
    .agg(
        Charlson_score_ever=("Weight", "sum"),
        Charlson_categories_ever=("Charlson_category", join_unique),
        n_charlson_categories_ever=("Charlson_category", "nunique"),
    )
)
patient_charlson_ever["Charlson_group_ever"] = pd.cut(
    patient_charlson_ever["Charlson_score_ever"],
    bins=[-1, 0, 2, 4, np.inf],
    labels=["0 (None)", "1-2 (Mild)", "3-4 (Moderate)", "5+ (Severe)"],
)

patient_demographics_with_ever_cci = patient_demographics.merge(patient_charlson_ever, on="DMRN", how="left")
patient_demographics_with_ever_cci["Charlson_score_ever"] = patient_demographics_with_ever_cci["Charlson_score_ever"].fillna(0).astype(int)
patient_demographics_with_ever_cci["Charlson_categories_ever"] = patient_demographics_with_ever_cci["Charlson_categories_ever"].fillna("None")
patient_demographics_with_ever_cci["n_charlson_categories_ever"] = patient_demographics_with_ever_cci["n_charlson_categories_ever"].fillna(0).astype(int)
patient_demographics_with_ever_cci["Charlson_group_ever"] = patient_demographics_with_ever_cci["Charlson_group_ever"].fillna("0 (None)")

print("mapped problem rows:", len(problem_charlson_mapped))
print("patients with any mapped Charlson condition:", problem_charlson_mapped["DMRN"].nunique())
print("patient_charlson_ever rows:", len(patient_charlson_ever))
print(patient_demographics_with_ever_cci["Charlson_group_ever"].value_counts(dropna=False).to_string())



## 7. Build Message-Level Charlson CCI At Message Time


In [ ]:
def build_message_level_charlson(
    messages: pd.DataFrame,
    problems_expanded: pd.DataFrame,
    message_id_col: str = "MESSAGE_ID",
    dmrn_col: str = "DMRN",
    time_col: str = "CREATED_TIME",
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Compute CCI for each message using diagnoses active at the message timestamp.

    A diagnosis is active for a message when:
    - NOTED_DATE is on or before CREATED_TIME, and
    - RESOLVED_DATE is missing or on/after CREATED_TIME.
    """
    demographic_cols = [
        col for col in ["BIRTH_YEAR", "Message_Year", "Age_at_Message", "AgeGroup65"]
        if col in messages.columns
    ]
    msg_subset = messages[[message_id_col, dmrn_col, time_col] + demographic_cols].copy()
    msg_subset[dmrn_col] = clean_id(msg_subset[dmrn_col])
    msg_subset[time_col] = to_ts(msg_subset[time_col])
    msg_subset = msg_subset.dropna(subset=[message_id_col, dmrn_col, time_col]).copy()

    problems = problems_expanded[["DMRN", "NOTED_DATE", "RESOLVED_DATE", "DX_CODE", "Charlson_category", "Weight"]].copy()
    problems["DMRN"] = clean_id(problems["DMRN"])
    problems["NOTED_DATE"] = to_ts(problems["NOTED_DATE"])
    problems["RESOLVED_DATE"] = to_ts(problems["RESOLVED_DATE"])
    problems["Weight"] = pd.to_numeric(problems["Weight"], errors="coerce").fillna(0).astype(int)

    merged = msg_subset.merge(problems, left_on=dmrn_col, right_on="DMRN", how="left", suffixes=("", "_problem"))
    active_mask = (
        (merged["NOTED_DATE"] <= merged[time_col])
        & (merged["RESOLVED_DATE"].isna() | (merged["RESOLVED_DATE"] >= merged[time_col]))
    )
    merged_active = merged.loc[active_mask, [message_id_col, "Charlson_category", "Weight"]].copy()
    merged_active = merged_active.dropna(subset=["Charlson_category"])
    merged_active = merged_active.drop_duplicates(subset=[message_id_col, "Charlson_category"])

    cci_msg = (
        merged_active.groupby(message_id_col, as_index=False)
        .agg(
            Charlson_score=("Weight", "sum"),
            Charlson_categories=("Charlson_category", join_unique),
        )
    )

    out = msg_subset.merge(cci_msg, on=message_id_col, how="left")
    out["Charlson_score"] = out["Charlson_score"].fillna(0).astype(int)
    out["Charlson_categories"] = out["Charlson_categories"].fillna("None")
    out["Charlson_group"] = pd.cut(
        out["Charlson_score"],
        bins=[-1, 0, 2, 4, np.inf],
        labels=["0 (None)", "1-2 (Mild)", "3-4 (Moderate)", "5+ (Severe)"],
    )

    qa = {
        "input_message_rows": int(len(messages)),
        "message_rows_used": int(len(msg_subset)),
        "messages_with_active_charlson_condition": int(cci_msg[message_id_col].nunique()),
        "active_message_condition_rows_after_dedup": int(len(merged_active)),
        "max_charlson_score": int(out["Charlson_score"].max()) if len(out) else 0,
        "charlson_group_counts": out["Charlson_group"].astype(str).value_counts(dropna=False).to_dict(),
    }
    return out, cci_msg, qa


## 8. Run Message-Level CCI


In [ ]:
message_charlson_cci, message_charlson_nonzero, message_cci_qa = build_message_level_charlson(
    messages=message_demographics,
    problems_expanded=problem_charlson_mapped,
    message_id_col=MESSAGE_ID_COL,
    dmrn_col=MESSAGE_DMRN_COL,
    time_col=MESSAGE_TIME_COL,
)


def json_safe_counts(series: pd.Series) -> dict:
    """Return value counts with JSON-safe string keys and integer values."""
    counts = series.value_counts(dropna=False)
    return {
        ("Missing" if pd.isna(key) else str(key)): int(value)
        for key, value in counts.items()
    }


message_cci_qa.update({
    "messages_with_birth_year": int(message_charlson_cci["BIRTH_YEAR"].notna().sum()),
    "messages_with_age_at_message": int(message_charlson_cci["Age_at_Message"].notna().sum()),
    "age_group65_counts": json_safe_counts(message_charlson_cci["AgeGroup65"]),
})

print("message_charlson_cci rows:", len(message_charlson_cci))
print("messages with birth year:", message_cci_qa["messages_with_birth_year"])
print("messages with age at message:", message_cci_qa["messages_with_age_at_message"])
print("messages with active Charlson condition:", message_cci_qa["messages_with_active_charlson_condition"])
print(message_charlson_cci["Charlson_group"].value_counts(dropna=False).to_string())
display(pd.DataFrame([message_cci_qa]))


## 9. Save Outputs


In [ ]:
patient_demographics_path = RUN_DIR / "patient_demographics_clean.parquet"
message_demographics_path = RUN_DIR / "message_demographics_clean.parquet"
message_charlson_cci_path = RUN_DIR / "message_level_charlson_cci.parquet"
qa_summary_path = RUN_DIR / "qa_demographics_cci_summary.json"

patient_demographics.to_parquet(patient_demographics_path, index=False)
message_demographics.to_parquet(message_demographics_path, index=False)
message_charlson_cci.to_parquet(message_charlson_cci_path, index=False)

# Canonical handoff copies used by downstream analysis-dataset construction.
MESSAGE_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
patient_demographics.to_parquet(CANONICAL_PATIENT_DEMOGRAPHICS_PATH, index=False)
message_demographics.to_parquet(CANONICAL_MESSAGE_DEMOGRAPHICS_PATH, index=False)
message_charlson_cci.to_parquet(CANONICAL_MESSAGE_CHARLSON_CCI_PATH, index=False)

def json_safe_counts(series: pd.Series) -> dict:
    """Create JSON-safe aggregate counts, including missing values."""
    counts = series.value_counts(dropna=False)

    return {
        ("Missing" if pd.isna(key) else str(key)): int(value)
        for key, value in counts.items()
    }

qa_summary = {
    "run_id": RUN_ID,
    "message_data_path": str(MESSAGE_DATA_PATH),
    "charlson_lookup_path": str(CHARLSON_LOOKUP_PATH),
    "saved_outputs": {
        "patient_demographics_clean": str(patient_demographics_path),
        "message_demographics_clean": str(message_demographics_path),
        "message_level_charlson_cci": str(message_charlson_cci_path),
        "qa_demographics_cci_summary": str(qa_summary_path),
    },
    "canonical_handoff_outputs": {
        "patient_demographics_clean": str(CANONICAL_PATIENT_DEMOGRAPHICS_PATH),
        "message_demographics_clean": str(CANONICAL_MESSAGE_DEMOGRAPHICS_PATH),
        "message_level_charlson_cci": str(CANONICAL_MESSAGE_CHARLSON_CCI_PATH),
    },
    "patient_demographics_rows": int(len(patient_demographics)),
    "patient_demographics_patients": int(patient_demographics["DMRN"].nunique()),
    "rows_with_birth_year": int(patient_demographics["BIRTH_YEAR"].notna().sum()),
    "message_demographics_rows": int(len(message_demographics)),
    "messages_with_birth_year": int(message_cci_qa["messages_with_birth_year"]),
    "messages_with_age_at_message": int(message_cci_qa["messages_with_age_at_message"]),
    "problem_list_rows": int(len(problem_list_clean)),
    "problem_list_patients": int(problem_list_clean["DMRN"].nunique()),
    "charlson_lookup_rows": int(len(Charlson_comorbidity)),
    "problem_charlson_mapped_rows": int(len(problem_charlson_mapped)),
    "patients_with_any_mapped_charlson_condition": int(problem_charlson_mapped["DMRN"].nunique()),
    "message_charlson_cci_rows": int(len(message_charlson_cci)),
    "messages_with_active_charlson_condition": int(message_cci_qa["messages_with_active_charlson_condition"]),
    "gender_counts": json_safe_counts(
        patient_demographics["Gender"]
    ),
    "race_group_counts": json_safe_counts(
        patient_demographics["Race_Grouped"]
    ),
    "ethnicity_group_counts": json_safe_counts(
        patient_demographics["Ethnicity_Grouped"]
    ),
    "message_charlson_group_counts": json_safe_counts(
        message_charlson_cci["Charlson_group"]
    ),
    "message_age_group65_counts": json_safe_counts(
        message_charlson_cci["AgeGroup65"]
    ),
}

with open(qa_summary_path, "w", encoding="utf-8") as f:
    json.dump(qa_summary, f, indent=2, default=str)

print("Saved demographics/CCI outputs:")
for path in [
    patient_demographics_path,
    message_demographics_path,
    message_charlson_cci_path,
    qa_summary_path,
]:
    print(" -", path)

print("Saved canonical downstream handoff outputs:")
for path in [
    CANONICAL_PATIENT_DEMOGRAPHICS_PATH,
    CANONICAL_MESSAGE_DEMOGRAPHICS_PATH,
    CANONICAL_MESSAGE_CHARLSON_CCI_PATH,
]:
    print(" -", path)

## Expected Outputs

Every run writes to a unique folder under `runs/demographics_cci_YYYYMMDD_HHMMSS/`.

Working outputs:

- `patient_demographics_clean.parquet`: one row per patient with cleaned gender, race, ethnicity, grouped demographic fields, and birth year.
- `message_demographics_clean.parquet`: one row per message with `MESSAGE_ID`, `DMRN`, `CREATED_TIME`, `BIRTH_YEAR`, `Message_Year`, `Age_at_Message`, and `AgeGroup65`.
- `message_level_charlson_cci.parquet`: one row per message with message-level age fields plus `Charlson_score`, `Charlson_categories`, and `Charlson_group` calculated from diagnoses active at the message timestamp.
- `qa_demographics_cci_summary.json`: aggregate QA counts for demographics, age, problem-list mapping, and message-time CCI.

Intermediate problem-list and Charlson-mapping tables are used in memory to reproduce the final variables but are not saved as additional row-level outputs.
